# `df.style`

possible API:

We'd like to conditionally format a dataframe, to style cells according to

- The cell's value
- The cell's value relative to the column / row / table
- The cell's position in the column / row / table

(these may be incompatible... not sure).

The API could be somthing like

```python
df.style(rows, columns, table)
```

All of which are optional. The values passed in should be

- function
- [function]
- dictionary :: {row/col -> [functions]}

The last one is just for `rows` and `columns`. It's used if you want to apply a certain style to a single or subset of columns.

What's the API for each `func`?

```python
def f(values, ...)
    pass
```

where `values` is either a Series for `rows` or `columns`, or a DataFrame for `table`.

Each function should return a Series/DataFrame shaped like its `values` input. The contents of the Series/DataFrame are css attribute: value pairs.

These styles are applied when we generate the HTML.

This should cover most cases. The only other thing is *maybe* the index / columns. Optionally allow for `index=` and `header=`. Maybe.

I think this strikes a reasonable balance between complexity / flexibility. One drawback is that writing and using generic styling functions is a bit awkward. You'll need to use `lambda`s or `partial` the functions before passing them into style if you want to adjust kwargs to the style functions.

# Background

@yp did a bunch of work on this in https://github.com/pydata/pandas/issues/3190.
I'm using his idea and much of his implementation of using Jinja templates to render translate from the DataFrame with styles to HTML.

In [1]:
from pandas.core.format import StyleFormatter

In [2]:
from jinja2 import Template, Environment
from matplotlib.colors import rgb2hex
import pandas.core.common as com
from IPython.display import HTML

np.random.seed(24)
df = pd.DataFrame({'A': np.linspace(1, 10, 10)})
df = pd.concat([df, pd.DataFrame(np.random.randn(10, 4), columns=list('BCDE'))],
               axis=1)
df

,A,B,C,D,E
0,1,1.329212,-0.770033,-0.316280,-0.990810
1,2,-1.070816,-1.438713,0.564417,0.295722
2,3,-1.626404,0.219565,0.678805,1.889273
3,4,0.961538,0.104011,-0.481165,0.850229
4,5,1.453425,1.057737,0.165562,0.515018
5,6,-1.336936,0.562861,1.392855,-0.063328
6,7,0.121668,1.207603,-0.002040,1.627796
7,8,0.354493,1.037528,-0.385684,0.519818
8,9,1.686583,-1.325963,1.428984,-2.089354
9,10,-0.129820,0.631523,-0.586538,0.290720


A few example styling functions. Perhaps we'll have a repo of these somewhere.

In [3]:
def highlight_null(s):
    # this should be a table function, but that argument handler is broken right now
    colors = list(np.where(s.isnull(), 'red', ''))
    return pd.Series([('background-color', color) for color in colors], index=s.index)

def color_bg_range(s, cmap='PuBu'):
    """Color background in a range according to the data."""
    normed = (s - s.min()) / (s.max() - s.min())
    colors = [rgb2hex(x) for x in plt.cm.get_cmap(cmap)(normed)]
    return pd.Series([('background-color', color) for color in colors], index=s.index)

def color_font_even(s, even_color='black', odd_color='red'):
    """"""
    colors = [even_color if x % 2 else odd_color for x in s]
    return pd.Series([('font-color', color) for color in colors], index=s.index)

def color_gradient(s, cmap='YlGn'):
    "Color background by position"
    colors = [rgb2hex(x) for x in plt.cm.get_cmap(cmap)(1 / np.arange(len(s)))]
    return pd.Series([('background-color', color) for color in colors], index=s.index)

def striped(s):
    colors = ["B5DEFF" if i % 2 else "" for i in range(len(s))]
    return pd.Series([('background-color', color) for color in colors], index=s.index)

def checker(table):
    nrow, ncol = table.shape
    colors = [[('background-color', 'red') if r % 2 or c % 2 else ('background-color', 'black')
               for r in range(ncol)]
              for c in range(nrow)]
    return pd.DataFrame(colors, index=table.index, columns=table.columns)


In [4]:
# note: users won't interact directly with StyleFormatter
# they'll just do `df.style(columns=color_bg_range)` and we'll return a _repr_html_ thing
s = StyleFormatter(df, columns=color_bg_range)
HTML(s.render())

/Users/tom.augspurger/Envs/dev/lib/python3.4/site-packages/pandas/pandas/core/index.py:4281: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return np.sum(name == np.asarray(self.names)) > 1


Adjust the `style` functions kwargs with partial:

In [5]:
from functools import partial

In [6]:
s = StyleFormatter(df, columns=partial(color_bg_range, cmap='YlGn'))
HTML(s.render())

/Users/tom.augspurger/Envs/dev/lib/python3.4/site-packages/pandas/pandas/core/index.py:4281: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return np.sum(name == np.asarray(self.names)) > 1


In [7]:
df = pd.DataFrame(np.random.randn(10, 10))
df.iloc[0, 3] = np.nan
df.iloc[1, 8] = np.nan
df.iloc[6, 4] = np.nan
df.iloc[8, 2] = np.nan

s = StyleFormatter(df, columns=highlight_null)
HTML(s.render())

/Users/tom.augspurger/Envs/dev/lib/python3.4/site-packages/pandas/pandas/core/index.py:4281: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  return np.sum(name == np.asarray(self.names)) > 1
